# **추천시스템을 만들어 봅시다!**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

# Load Data
fname_tran = '../input/h-and-m-personalized-fashion-recommendations/transactions_train.csv'
fname_article = '../input/h-and-m-personalized-fashion-recommendations/articles.csv'

# CSV 파일 로드
data = pd.read_csv(fname_tran, usecols=['customer_id', 'article_id', 'price'])
articles = pd.read_csv(fname_article)

# 🔹 article_id를 문자열로 변환 (데이터 일관성 유지)
data['article_id'] = data['article_id'].astype(str)
articles['article_id'] = articles['article_id'].astype(str)

# 🔹 이미지 경로 생성 (H&M 데이터셋 폴더 구조 적용)
articles['image_path'] = articles['article_id'].apply(lambda x: f"0{x[:2]}/0{x}.jpg")

# 🔹 데이터 가공 (고객-상품 매핑)
data['count'] = 1
data = data.groupby(['customer_id', 'article_id'], as_index=False).sum()
user_unique = data['customer_id'].unique()
article_unique = data['article_id'].unique()

# 🔹 ID 변환 (Mapping)
user_to_idx = {v: k for k, v in enumerate(user_unique)}
article_to_idx = {v: k for k, v in enumerate(article_unique)}
idx_to_article = {v: k for k, v in article_to_idx.items()}

# 🔹 ID 매핑 (NaN 방지 및 필터링)
data['customer_id'] = data['customer_id'].map(user_to_idx).fillna(-1).astype(int)
data['article_id'] = data['article_id'].map(article_to_idx).fillna(-1).astype(int)
data = data[(data['customer_id'] >= 0) & (data['article_id'] >= 0)]

# 🔹 CSR Matrix 생성
num_user = data['customer_id'].nunique()
num_article = data['article_id'].nunique()
csr_data = csr_matrix((data['count'], (data.customer_id, data.article_id)), shape=(num_user, num_article))

# 🔹 ALS 모델 학습
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['MKL_NUM_THREADS'] = '1'

als_model = AlternatingLeastSquares(factors=360, regularization=0.01, use_gpu=True, iterations=5, dtype=np.float32, calculate_training_loss=True)
als_model.fit(csr_data.T)

# 🔹 추천 시스템 - 이미지 갤러리 표시
def show_recommendations(input_id, is_user=True, N=10):
    """ 사용자가 구매할 만한 추천 아이템 또는 특정 제품과 유사한 제품을 갤러리로 표시 """
    if is_user:
        if input_id not in user_to_idx:
            print("사용자 ID를 찾을 수 없습니다.")
            return
        user_idx = user_to_idx[input_id]
        recommended = als_model.recommend(user_idx, csr_data, N=N)
        recommended_articles = [idx_to_article[i[0]] for i in recommended]
    else:
        if input_id not in article_to_idx:
            print("상품 ID를 찾을 수 없습니다.")
            return
        article_idx = article_to_idx[input_id]
        similar_items = als_model.similar_items(article_idx, N=N)
        recommended_articles = [idx_to_article[i[0]] for i in similar_items]
    
    # 추천 아이템 정보 가져오기
    recommended_df = articles[articles['article_id'].isin(recommended_articles)]
    
    # 🔹 이미지 갤러리 출력
    fig, axes = plt.subplots(1, len(recommended_df), figsize=(15, 5))
    if len(recommended_df) == 1:
        axes = [axes]  # 리스트로 변환
    for ax, (_, row) in zip(axes, recommended_df.iterrows()):
        img_path = f"../input/h-and-m-personalized-fashion-recommendations/images/{row['image_path']}"
        
        try:
            img = plt.imread(img_path)
            ax.imshow(img)
        except FileNotFoundError:
            ax.text(0.5, 0.5, "No Image", fontsize=12, ha='center', va='center')

        ax.set_title(row['prod_name'][:15])
        ax.axis("off")
    plt.show()

# 사용 예시
# show_recommendations('000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318', is_user=True)
# show_recommendations('176209023', is_user=False)

In [ ]:
show_recommendations('000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318', is_user=True)
# show_recommendations(176209023, is_user=False)

In [ ]:
show_recommendations('176209023', is_user=False)